# 02 探索性分析與縣市相關熱圖

**用途**：以階層式分群重新排序縣市相關矩陣並畫熱圖；繪製各縣市折線圖與箱型圖，觀察水準與趨勢。

| 項目 | 說明 |
|---|---|
| 輸入 | pivot_table_age_std、corr_matrix_county |
| 輸出 | 縣市相關熱圖、折線圖、箱型圖（存於 notebooks/_scratch/） |
| 來源 | MVA 期末報告分析.ipynb（Colab 原檔，191 cells） 之 cell 108–115（Update 05/29 最終版） |


In [ ]:
# 共用設定：載入資料並提供 df / pivot_table_age_std / pivot_table_cleaned / scaled_data / taiwan_map / prop 等變數
# （取代原 Colab 的 Drive 掛載、字型下載與 GDAL 設定；詳見 _legacy_setup.py）
import sys; sys.path.insert(0, '..')   # _legacy_setup.py 位於 notebooks/
from _legacy_setup import *

In [ ]:
# 前置狀態（原 Colab 於 01 產生）
corr_matrix_county = pivot_table_age_std.T.corr()   # 縣市 × 縣市
corr_matrix_yr = pivot_table_age_std.corr()          # 年份 × 年份

## Correlation Visualization

* Visualize the reordered correlation matrix (counties) using a heatmap.
* 這裡只放「縣市間」的相關。

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt
import seaborn as sns

# Perform hierarchical clustering
linked = sch.linkage(corr_matrix_county, method='ward')

# Get the order of the clusters
cluster_order = sch.dendrogram(linked, no_plot=True)['leaves']

# Reorder the correlation matrix based on the clustering result
reordered_corr_matrix_county = corr_matrix_county.iloc[cluster_order, cluster_order]
display(reordered_corr_matrix_county.head())

In [ ]:
plt.figure(figsize=(12, 10))

ax = sns.heatmap(
    reordered_corr_matrix_county,
    annot=True,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    fmt=".2f",
    linewidths=.5,
    annot_kws={'fontsize': 10}
)

plt.title('各縣市年齡標準化發生率相關係數矩陣（依 Hclust 排序）')

plt.savefig('correlation_heatmap_counties.png', dpi=300, bbox_inches='tight')

plt.show()

## EDA (肺癌發生率)

* 折線圖
* 箱型圖

In [ ]:
# 計算字型大小 (依你的需求放大 10%)
base_size = 12 * 1.1
title_size = 16 * 1.1
label_size = 14 * 1.1

plt.figure(figsize=(14, 8))
sns.lineplot(data=df_filtered, x='Diagnosis_Year', y='Age_Standardized_Incidence_Rate', hue='City_County')

# 內建全自動套用中文字型，直接帶入大小即可！
plt.title('肺癌 — 年齡標準化發生率 — 按縣市與年度', fontsize=title_size)
plt.xlabel('年', fontsize=label_size)
plt.ylabel('年齡標準化發生率 (每10萬人口)', fontsize=label_size)

# 圖例也一樣，完全不需 setp 跑迴圈
plt.legend(title='縣市別', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=base_size, title_fontsize=label_size)

plt.grid(True)
plt.tight_layout()
plt.savefig('lung_cancer_lineplot_counties.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure Chinese font is applied
if 'prop' in locals() or 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

plt.figure(figsize=(16, 8))
sns.boxplot(data=df_filtered, x='City_County', y='Age_Standardized_Incidence_Rate')
plt.title('肺癌 — 年齡標準化發生率 — 按縣市別 (箱型圖)', fontsize=title_size)
plt.xlabel('縣市別', fontsize=label_size)
plt.ylabel('年齡標準化發生率 (每10萬人口)', fontsize=label_size)
plt.xticks(rotation=45, ha='right', fontsize=base_size)
plt.yticks(fontsize=base_size)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('lung_cancer_boxplot_counties.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure Chinese font is applied
if 'prop' in locals() or 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

plt.figure(figsize=(16, 8))

# ✨ 關鍵步驟：計算各縣市的中位數，由小到大排序，並取出縣市名稱 (Index)
county_order = (
    df_filtered.groupby('City_County')['Age_Standardized_Incidence_Rate']
    .median()
    .sort_values()
    .index
)

# ✨ 在 boxplot 中加入 order=county_order
sns.boxplot(
    data=df_filtered,
    x='City_County',
    y='Age_Standardized_Incidence_Rate',
    order=county_order,
    hue='City_County',
    hue_order=county_order,
    palette='coolwarm',
    legend=False
)
sns.despine()

plt.title('肺癌 — 年齡標準化發生率 — 按縣市別 (箱型圖，依中位數排序)', fontsize=title_size)
plt.xlabel('縣市別', fontsize=label_size)
plt.ylabel('年齡標準化發生率 (每10萬人口)', fontsize=label_size)

# 因為依序排列了，標籤拉直或 45 度都會看起來非常整齊
plt.xticks(rotation=45, ha='right', fontsize=base_size)
plt.yticks(fontsize=base_size)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('lung_cancer_boxplot_counties_sorted.png', dpi=300, bbox_inches='tight')
plt.show()

城鄉與地理分布趨勢（低 $\rightarrow$ 高）
* 低發生率群組（左側）：金門縣的中位數明顯最低（約在 16 左右），接著是苗栗縣、屏東縣。
* 高發生率群組（右側）：發生率中位數最高的幾個縣市高度集中在新北市、基隆市、宜蘭縣、彰化縣、台中市。這很符合一般對於人口稠密、工業化程度較高、或特定環境因子的印象。
---
IQR
* 最小變異：台北市
* 波動極大的縣市：連江縣、澎湖縣、雲林縣
* 高度穩定的縣市：例如花蓮縣、新竹市、台北市。她們的箱子很扁，代表這些縣市在過去這些年來，肺癌年齡標準化發生率非常穩定，沒有太大的暴增或暴跌。
* 波動極大的縣市：例如澎湖縣、雲林縣、嘉義市。她們的箱子拉得非常長（尤其是雲林縣，下限快到 5，上限快到 45）。這暗示這些縣市的發生率可能在某些特定年份有劇烈的環境轉變、政策篩檢推動、或是資料統計上的大波動。
---
Outliers
* 連江縣與澎湖縣的上限極端值：這兩個離島（特別是連江與澎湖）都有一個往上飆高非常多的點。因為離島人口基數小，「年齡標準化發生率」（每10萬人口）容易因為某一年突然多了幾位確診者，就導致數值暴衝。這在統計上是很典型的**小樣本效應**。
* 台北市與基隆市的極端值：台北市有一個極高值和一個極低值；基隆市則有一個極低值。